# Part 1 — Explore the Dataset

**Goal:** Load the OASIS3 PIB PET data, inspect its structure, and produce a summary of subjects and measurements.

---

## Getting started in VS Code (first time with Jupyter)

Jupyter notebooks let you run code one block at a time and see output immediately beneath each block — great for exploring data step by step.

**One-time setup:**
1. Open this file in VS Code — it should open as a notebook automatically.
2. VS Code will prompt you to install the **Jupyter** extension if it isn't already — click **Install**.
3. In the top-right corner, click **Select Kernel → Python Environments** and choose the `.venv` environment you created during setup. If you haven't set up the environment yet, follow the steps in the README first.

**Running cells:**
- Click the **▶ Run Cell** button to the left of any code block, or press **Shift + Enter** to run and advance to the next cell.
- Run cells **in order from top to bottom** — each one builds on the previous.
- Output (printed text, plots, errors) appears directly below the cell.

---

## Available helper functions

All helpers live in `utils/oasis_utils.py`. **Read through that file before starting** — the docstrings tell you exactly what each function expects and returns. Here is a quick reference:

| Function | What it does |
|----------|-------------|
| `load_dataset(path)` | Loads an Excel or CSV file → returns a DataFrame |
| `validate_columns(df, cols)` | Raises an error if any column in `cols` is missing from `df` |
| `add_extracted_subject_id(df, id_col)` | Parses `'OAS30001'` from `'OAS30001_PIB_..._d0423'` → adds `Subject_ID_Extracted` column |
| `add_days_since_start_from_id(df, id_col, output_days, output_years)` | Parses day offset from the ID string, converts to years → adds two new columns |
| `ensure_dir(path)` | Creates a directory if it does not already exist |

---

## Your tasks

Work through **TODO 1 → TODO 6** in order. Each cell has detailed instructions.  
**When you finish a TODO, delete the `raise NotImplementedError(...)` line in that cell.**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure the utils folder is importable
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from utils.oasis_utils import (
    load_dataset,
    validate_columns,
    add_extracted_subject_id,
    add_days_since_start_from_id,
    ensure_dir,
)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH   = os.path.join("data", "OASIS3_PUP.xlsx")
RESULTS_DIR = "results"

TRACER   = "PIB"
ID_COL   = "PUP_PUPTIMECOURSEDATA ID"
SUVR_COL = "Centil_fSUVR_TOT_CORTMEAN"   # Centiloid (non-RSF)

ensure_dir(RESULTS_DIR)
print("Setup complete.")

---
## TODO 1 — Load the dataset

Use `load_dataset()` to load the data file and store it in a variable called `df`.

```
load_dataset(path)  →  DataFrame
```
- `path`: a file path string — use `DATA_PATH` which is already defined above.

After loading, call `df.head()` on the last line of the cell so the first few rows display automatically below — this is how you confirm the file loaded correctly.

In [ ]:
# YOUR CODE HERE
# df = ...

raise NotImplementedError("TODO 1: load the dataset")

df.head()  # Leave this here — it will display the first 5 rows once df is defined

---
## TODO 2 — Filter to PIB tracer and drop missing values

**a)** Validate that the required columns exist in `df`:
```python
validate_columns(df, [ID_COL, SUVR_COL, 'tracer'])
```
This will raise a clear error if any column is missing — useful for catching typos early.

**b)** Keep only rows where `'tracer'` equals `TRACER` (`'PIB'`).

**c)** Drop rows where `ID_COL` or `SUVR_COL` is `NaN`:
```python
df.dropna(subset=[...])
```

Confirm it worked by calling `df.head()` — you should see only PIB rows.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError("TODO 2: filter to PIB and drop NaNs")

df.head()  # Should show only PIB rows once complete

---
## TODO 3 — Add a simplified subject ID column

Use `add_extracted_subject_id()` to parse the short subject ID from the full measurement ID string.

```
add_extracted_subject_id(df, id_col)  →  DataFrame
```
- `df`: your DataFrame
- `id_col`: the column containing the full measurement IDs → use `ID_COL`

This adds a new column called `Subject_ID_Extracted` to `df`.  
Example: `'OAS30001_PIB_PUPTIMECOURSE_d0423'` → `'OAS30001'`

Confirm the new column looks right — call `df.head()` and check that `Subject_ID_Extracted` contains short IDs like `OAS30001`.

In [ ]:
# YOUR CODE HERE
# df = add_extracted_subject_id(df, ...)

raise NotImplementedError("TODO 3: add extracted subject ID")

df.head()  # Check that Subject_ID_Extracted appears and looks correct

---
## TODO 4 — Add days and years since first scan

Use `add_days_since_start_from_id()` to parse the time offset embedded in the measurement ID and convert it to years.

```
add_days_since_start_from_id(df, id_col, output_days, output_years)  →  DataFrame
```
- `df`: your DataFrame
- `id_col`: the full measurement ID column → use `ID_COL`
- `output_days`: name for the new days column → use `'Days'`
- `output_years`: name for the new years column → use `'Years'`

Confirm both new columns appear — call `df.head()` and check that `Days` and `Years` have sensible values (e.g. subject at day 0 on their first scan).

In [ ]:
# YOUR CODE HERE
# df = add_days_since_start_from_id(df, ..., output_days=..., output_years=...)

raise NotImplementedError("TODO 4: add days/years columns")

df.head()  # Check that Days and Years columns appear with sensible values

---
## TODO 5 — Print summary statistics

Print the following to the cell output:

```
Total subjects     : <number of unique Subject_ID_Extracted values>
Total measurements : <total number of rows>

Centiloid statistics (all measurements):
  min   : <value>
  max   : <value>
  mean  : <value>
  median: <value>
```

Useful methods:
- `df['Subject_ID_Extracted'].nunique()` — count of unique subjects
- `df[SUVR_COL].describe()` — returns a Series; access values with `['min']`, `['max']`, `['mean']`, `['50%']`

> **Something to think about:** Notice the gap between the mean and median Centiloid values. What does a large difference between them suggest about the shape of the distribution? What might that mean for this population?

In [ ]:
# YOUR CODE HERE

raise NotImplementedError("TODO 5: print summary statistics")

---
## TODO 6 — Plot a histogram of measurements per subject

**Steps:**

**a)** Count how many measurements each subject has:
```python
counts = df.groupby('Subject_ID_Extracted').size()
```

**b)** Use `np.histogram()` to compute bin counts and edges — this gives you fine-grained control over bin placement:
```python
hist_counts, bins = np.histogram(counts, bins=range(1, counts.max() + 2))
```

**c)** Plot using `plt.bar()` with the bin edges and counts:
```python
plt.bar(bins[:-1], hist_counts, width=np.diff(bins), edgecolor='black', align='edge')
```

**d)** Label the axes (`"Measurements per Subject"`, `"Number of Subjects"`), add a title, and remove the top and right spines for a clean look:
```python
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
```

**e)** Call `plt.tight_layout()` then save to `results/measurements_per_subject.png` at `dpi=200`.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError("TODO 6: plot histogram of measurements per subject")

---
## Reflection questions

Answer these in a new markdown cell below (double-click any markdown cell to edit it, then **Shift + Enter** to render).

**Q1.** How many subjects have only a single PIB scan? Why might these subjects be less useful for studying progression?

**Q2.** What does a Centiloid value of 0 represent? What about 100? *(Hint: look up the Centiloid standardisation paper.)*

**Q3.** You noted a large gap between the mean and median Centiloid values. What does that tell you about the distribution of amyloid burden in this dataset? What might it mean for interpreting the mean alone as a summary statistic?